## 1. Fetch the data from W&B

We start a W&B run in the `eda` group so this exploration is tracked alongside the pipeline runs. `save_code=True` uploads and versions this notebook in W&B. Using the `:latest` alias pins the exact artifact version the notebook consumed, which keeps the analysis traceable in the artifact lineage.

In [1]:
import wandb
import pandas as pd

# Start a W&B run so this EDA is versioned together with the pipeline
run = wandb.init(project="nyc_airbnb", group="eda", save_code=True)

# Download the raw sample produced by the "download" step
local_path = wandb.use_artifact("sample.csv:latest").file()
df = pd.read_csv(local_path)

df.head()

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Find your API key here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /home/student/.netrc
wandb: Currently logged in as: reyrosales (reyrosales-macewan-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365
0,2539,Clean & quiet apt home by the park,2787,John,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,1,9,2018-10-19,0.21,6,365
1,2595,Skylit Midtown Castle,2845,Jennifer,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,1,45,2019-05-21,0.38,2,355
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,4632,Elisabeth,Manhattan,Harlem,40.80902,-73.94190,Private room,150,3,0,NaN,NaN,1,365
3,3831,Cozy Entire Floor of Brownstone,4869,LisaRoxanne,Brooklyn,Clinton Hill,40.68514,-73.95976,Entire home/apt,89,1,270,2019-07-05,4.64,1,194
4,5022,Entire Apt: Spacious Studio/Loft by central park,7192,Laura,Manhattan,East Harlem,40.79851,-73.94399,Entire home/apt,80,10,9,2018-11-19,0.10,1,0


In [2]:
# Basic shape and column types
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
df.info()

Rows: 48895, Columns: 16
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48895 entries, 0 to 48894
Data columns (total 16 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   id                              48895 non-null  int64  
 1   name                            48879 non-null  object 
 2   host_id                         48895 non-null  int64  
 3   host_name                       48874 non-null  object 
 4   neighbourhood_group             48895 non-null  object 
 5   neighbourhood                   48895 non-null  object 
 6   latitude                        48895 non-null  float64
 7   longitude                       48895 non-null  float64
 8   room_type                       48895 non-null  object 
 9   price                           48895 non-null  int64  
 10  minimum_nights                  48895 non-null  int64  
 11  number_of_reviews               48895 non-null  int64  
 12  last_re

## 2. Profile report

ydata-profiling (the maintained successor of pandas-profiling) gives a quick overview of every column: distributions, missing values, correlations and warnings.

In [3]:
import ydata_profiling

profile = ydata_profiling.ProfileReport(df)
profile.to_notebook_iframe()

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 16/16 [00:01<00:00, 11.84it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

## 3. Findings from the profile

The report highlights three issues that matter for modeling.

### 3.1 Missing values
Several columns contain missing values (`name`, `host_name`, `last_review`, `reviews_per_month`). The missing values in `last_review` and `reviews_per_month` occur together, because listings that were never reviewed have neither a last review date nor a review rate.

In [4]:
# Count and share of missing values per column (only columns with missing values)
missing = df.isna().sum()
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({"missing": missing, "percent": missing_pct}).query("missing > 0")

,missing,percent
name,16,0.03
host_name,21,0.04
last_review,10052,20.56
reviews_per_month,10052,20.56


**Decision: we do not impute missing values here.** New data arriving in production will contain the same kinds of gaps, so imputation has to be part of the inference pipeline (the scikit-learn pipeline exported with the model). If we imputed here, the model would be trained on data that looks different from what it receives at prediction time, and the imputation logic would not be versioned with the model. The `train_random_forest` step therefore handles missing values with `SimpleImputer` steps inside the model pipeline.

### 3.2 `last_review` is stored as text
The `last_review` column is read as a string (`object`) instead of a date.

In [5]:
# last_review is stored as a string, not as a datetime
print(df["last_review"].dtype)
df["last_review"].dropna().head()

object


0    2018-10-19
1    2019-05-21
3    2019-07-05
4    2018-11-19
5    2019-06-22
Name: last_review, dtype: object

### 3.3 Price outliers
The `price` column has a long right tail with extreme values and also contains zeros, which are not realistic nightly prices.

In [6]:
# Summary statistics of price, including extreme quantiles
df["price"].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99])

count    48895.000000
mean       152.720687
std        240.154170
min          0.000000
1%          30.000000
5%          40.000000
50%        106.000000
95%        355.000000
99%        799.000000
max      10000.000000
Name: price, dtype: float64

In [7]:
# Number of listings that fall outside the range agreed with stakeholders
print("Price == 0:", (df["price"] == 0).sum())
print("Price < 10:", (df["price"] < 10).sum())
print("Price > 350:", (df["price"] > 350).sum())

Price == 0: 11
Price < 10: 11
Price > 350: 2456


After discussing with stakeholders, the accepted nightly price range is **$10 to $350, inclusive**. Listings outside that range are treated as outliers and removed, since the business case is estimating typical prices.

## 4. Apply the basic fixes

1. Keep only rows with `10 <= price <= 350` (`Series.between` is inclusive on both ends).
2. Convert `last_review` to a datetime.

These are exactly the operations the `basic_cleaning` component reimplements, with `min_price` and `max_price` read from `config.yaml`.

In [8]:
# Drop price outliers (inclusive range)
min_price = 10
max_price = 350
idx = df["price"].between(min_price, max_price)
df = df[idx].copy()

# Convert last_review to datetime (missing values become NaT, they are not imputed)
df["last_review"] = pd.to_datetime(df["last_review"])

## 5. Verify the fixes

In [9]:
# Prices must now be inside the accepted range and last_review must be a datetime
df.info()
print("Price range after cleaning:", df["price"].min(), "to", df["price"].max())
assert df["price"].between(min_price, max_price).all()
assert pd.api.types.is_datetime64_any_dtype(df["last_review"])

<class 'pandas.core.frame.DataFrame'>
Index: 46428 entries, 0 to 48894
Data columns (total 16 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   id                              46428 non-null  int64         
 1   name                            46413 non-null  object        
 2   host_id                         46428 non-null  int64         
 3   host_name                       46407 non-null  object        
 4   neighbourhood_group             46428 non-null  object        
 5   neighbourhood                   46428 non-null  object        
 6   latitude                        46428 non-null  float64       
 7   longitude                       46428 non-null  float64       
 8   room_type                       46428 non-null  object        
 9   price                           46428 non-null  int64         
 10  minimum_nights                  46428 non-null  int64         
 11  number_

In [10]:
# Missing values are still present on purpose: they are handled in the inference pipeline
df.isna().sum()[lambda s: s > 0]

name                   15
host_name              21
last_review          9182
reviews_per_month    9182
dtype: int64

Prices are now within the $10 to $350 range, `last_review` is a datetime, and the missing values remain for the inference pipeline to handle.

## 6. Finish the W&B run

In [11]:
run.finish()